In [ ]:
#سباامامیان-40315154
#در این پروژه با استفاده از الگوریتم زنبور، هدف ما تخصیص ۳۸۰ خانه به ۱۰ مدرسه (هر مدرسه ۳۸ خانه) به گونه‌ای بود که مجموع فاصله خانه‌ها تا مدارس به حداقل برسد. نتایج در پایان آمده‌اند.
# نصب پکیج‌های مورد نیاز:
# pip install geopandas shapely numpy scipy

import geopandas as gpd  # برای کار با داده‌های جغرافیایی (فایل‌های SHP)
import numpy as np       # برای محاسبات عددی
from scipy.spatial.distance import cdist  # برای محاسبه فاصله اقلیدسی بین نقاط
import random            # برای تولید اعداد تصادفی

# آدرس فایل‌های SHP (نقشه خانه‌ها و مدارس)
houses_path = r"C:/Users/Lenovo/Desktop/tamrin1/data/houses.shp"
schools_path = r"C:/Users/Lenovo/Desktop/tamrin1/data/schools.shp"

# گام ۱: خواندن داده‌های SHP با استفاده از geopandas
houses = gpd.read_file(houses_path)
schools = gpd.read_file(schools_path)

# گام ۲: تبدیل سیستم مختصات به UTM (متر) برای محاسبه دقیق فاصله
houses = houses.to_crs(epsg=3857)
schools = schools.to_crs(epsg=3857)

# گام ۳: استخراج مختصات مرکز هندسی خانه‌ها و مدارس
house_coords = np.array([(geom.centroid.x, geom.centroid.y) for geom in houses.geometry])
school_coords = np.array([(geom.centroid.x, geom.centroid.y) for geom in schools.geometry])

# گام ۴: محاسبه ماتریس فاصله اقلیدسی بین هر خانه و مدرسه (380 × 10)
distance_matrix = cdist(house_coords, school_coords, metric='euclidean')

# الگوریتم زنبور برای تخصیص خانه‌ها به مدارس بهینه

# پارامترها
num_houses = len(house_coords)      # تعداد کل خانه‌ها (مثلاً 380)
num_schools = len(school_coords)    # تعداد مدارس (مثلاً 10)
school_capacity = 38                # ظرفیت هر مدرسه (ثابت در صورت سؤال)
num_bees = 50                       # تعداد زنبورها (جواب‌های تصادفی اولیه)
num_iterations = 100               # تعداد تکرار الگوریتم

# تابع تولید جمعیت اولیه (تخصیص تصادفی ولی مطابق ظرفیت)
def initialize_population():
    population = []
    for _ in range(num_bees):
        solution = np.full(num_houses, -1)  # آرایه‌ای با اندازه تعداد خانه، مقدار اولیه -1
        house_indices = np.random.permutation(num_houses)  # ترتیب تصادفی خانه‌ها
        school_load = np.zeros(num_schools, dtype=int)     # ظرفیت مصرف‌شده برای هر مدرسه
        for h in house_indices:
            school_distances = distance_matrix[h]
            sorted_schools = np.argsort(school_distances)  # مدارس نزدیک‌تر اول
            for s in sorted_schools:
                if school_load[s] < school_capacity:
                    solution[h] = s
                    school_load[s] += 1
                    break
        population.append(solution)
    return population

# تابع هدف: جمع کل فاصله‌های تخصیص‌داده‌شده بین خانه و مدرسه مربوطه
def evaluate_solution(solution):
    return sum(distance_matrix[i][s] for i, s in enumerate(solution))

# اجرای اصلی الگوریتم زنبور
population = initialize_population()  # تولید جمعیت اولیه
best_solution = None
best_score = float('inf')

# تکرار الگوریتم
for iteration in range(num_iterations):
    scores = [evaluate_solution(sol) for sol in population]  # ارزیابی همه پاسخ‌ها
    sorted_indices = np.argsort(scores)                      # مرتب‌سازی بر اساس بهترین جواب
    population = [population[i] for i in sorted_indices]
    scores = [scores[i] for i in sorted_indices]

    if scores[0] < best_score:
        best_score = scores[0]
        best_solution = population[0].copy()  # ذخیره بهترین جواب

    # جستجوی محلی: جابجایی تصادفی بین دو خانه در بهترین جواب
    new_population = [best_solution]
    for i in range(1, num_bees):
        new_sol = best_solution.copy()
        h1, h2 = np.random.choice(num_houses, 2, replace=False)
        new_sol[h1], new_sol[h2] = new_sol[h2], new_sol[h1]
        new_population.append(new_sol)

    population = new_population  # بروزرسانی جمعیت برای تکرار بعدی

# نمایش نتایج نهایی
school_counts = np.bincount(best_solution, minlength=num_schools)  # تعداد خانه در هر مدرسه

print("\n--- نتایج نهایی ---")
for i, count in enumerate(school_counts):
    print(f"مدرسه {i+1}: {count} خانه")

print(f"\nمجموع کل فاصله‌ها: {best_score:.2f} متر")
